# OptiVision easy-mode validation, batch e4a_easy_dt_tables_figures (DT)

DT: ArxivQA, TabFQuAD, TAT-DQA, the four design models.

- `vidore/colpali-v1.3` on v1_arxivqa (500 pages), v1_tabfquad (280 pages), v1_tatdqa (1,663 pages): 1.6-1.7 h
- `vidore/colqwen2-v1.0-hf` on v1_arxivqa (500 pages), v1_tabfquad (280 pages), v1_tatdqa (1,663 pages): 0.5-0.7 h
- `vidore/colqwen2.5-v0.2` on v1_arxivqa (500 pages), v1_tabfquad (280 pages), v1_tatdqa (1,663 pages): 2.2-2.4 h
- `vidore/colSmol-500M` on v1_arxivqa (500 pages), v1_tabfquad (280 pages), v1_tatdqa (1,663 pages): 1.0-1.7 h

**Estimate:** 5.6-7.0 GPU hours (session limit 11 h).

**Before it may run:** prereg parts 1, 2 and 3 frozen and pushed (reports/easy/PREREG_EASY.frozen.json). The guard cell after the clone stops the notebook, before any install
or GPU work, if the `easy-mode` branch lacks `reports/easy/PREREG_EASY.frozen.json` with preregistration
part(s) 1, 2, 3, if any frozen file changed since, or if a required file is missing.

**Run it:** *Settings -> Accelerator*: **GPU T4 x2**, *Internet*: **On**. Then *Save Version -> Save & Run All
(Commit)*. Download `results_e4a_easy_dt_tables_figures.zip` from the version's *Output* tab and ingest it with
`python scripts/easy_validation/ingest.py results_e4a_easy_dt_tables_figures.zip`.

**Resume:** if the session stopped at its deadline (`batch_summary.json` lists a job as `partial`), add this
notebook's output as input (*Add data -> Notebook output*) and run it again: finished shards are kept.

In [ ]:
import json as _json
BRANCH = "easy-mode"
NAME = "e4a_easy_dt_tables_figures"
JOBS = _json.loads("[{\"corpora\": [\"v1_arxivqa\", \"v1_tabfquad\", \"v1_tatdqa\"], \"dtype\": \"float32\", \"keep_float32\": [], \"kind\": \"image\", \"model\": \"vidore/colpali-v1.3\", \"multi_gpu\": true, \"role\": \"encode\"}, {\"batch_size\": 2, \"corpora\": [\"v1_arxivqa\", \"v1_tabfquad\", \"v1_tatdqa\"], \"dtype\": \"float16\", \"keep_float32\": [], \"kind\": \"image\", \"model\": \"vidore/colqwen2-v1.0-hf\", \"role\": \"encode\"}, {\"corpora\": [\"v1_arxivqa\", \"v1_tabfquad\", \"v1_tatdqa\"], \"dtype\": \"float32\", \"keep_float32\": [], \"kind\": \"image\", \"model\": \"vidore/colqwen2.5-v0.2\", \"multi_gpu\": true, \"role\": \"encode\"}, {\"corpora\": [\"v1_arxivqa\", \"v1_tabfquad\", \"v1_tatdqa\"], \"dtype\": \"float32\", \"keep_float32\": [], \"kind\": \"image\", \"model\": \"vidore/colSmol-500M\", \"multi_gpu\": true, \"role\": \"encode\"}]")
EXTRA_PIP = []
DEADLINE_HOURS = 11.0
REQUIRED_PARTS = ["1", "2", "3"]
REQUIRED_FILES = [
 "scripts/easy_validation/dt.py",
 "scripts/easy_validation/encode_unit.py",
 "scripts/easy_validation/units.py",
 "scripts/encode_text_vectors.py",
 "scripts/encode_vectors.py"
]
REQUIRED_FROZEN = []

In [ ]:
import json, os, shutil, subprocess, sys
if shutil.which("nvidia-smi") is None:
    raise SystemExit("NO GPU IN THIS SESSION. Settings (right panel) -> Accelerator -> GPU T4 x2, then Save Version "
                     "-> Save & Run All (Commit) again.")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
WORK = "/kaggle/working"
REPO = f"{WORK}/optivision-rag"
RESULTS = f"{WORK}/results_{NAME}"
os.environ["HF_HOME"] = "/tmp/hf"
!rm -rf {REPO} && git clone -q --depth 1 --branch {BRANCH} https://github.com/Daemon-VI/optivision-rag.git {REPO}
!cd {REPO} && git log --oneline -1

In [ ]:
# Preregistration guard (reports/easy/PREREG_EASY.md): stop here, before any install or GPU work,
# unless the cloned branch carries the frozen preregistration this batch needs, unchanged.
import hashlib, json, pathlib
def _digest(p):
    return hashlib.sha256(p.read_bytes().replace(b"\r\n", b"\n")).hexdigest()
_repo = pathlib.Path(REPO)
_frozen = _repo / "reports" / "easy" / "PREREG_EASY.frozen.json"
_problems = []
if not _frozen.exists():
    _problems.append("reports/easy/PREREG_EASY.frozen.json is missing: the preregistration is not frozen")
else:
    _record = json.loads(_frozen.read_text(encoding="utf-8"))
    _parts = _record.get("parts", {})
    _problems += [f"preregistration part {p} is not frozen" for p in REQUIRED_PARTS if str(p) not in _parts]
    _listed = {}
    for _entry in _parts.values():
        _listed.update(_entry.get("files", {}))
    for _rel, _sha in sorted(_listed.items()):
        _p = _repo / _rel
        if not _p.exists():
            _problems.append(f"frozen file {_rel} is missing")
        elif _digest(_p) != _sha:
            _problems.append(f"{_rel} changed after it was frozen")
        elif _rel.endswith(".md") and "NOT FROZEN" in _p.read_text(encoding="utf-8"):
            _problems.append(f"{_rel} is still a draft")
    _problems += [f"{_rel} is not among the frozen files" for _rel in REQUIRED_FROZEN if _rel not in _listed]
_problems += [f"{_rel} is missing from the {BRANCH} branch" for _rel in REQUIRED_FILES if not (_repo / _rel).exists()]
if _problems:
    raise SystemExit("REFUSING TO RUN: nothing was installed or encoded.\n- " + "\n- ".join(_problems))
print(f"preregistration parts {list(REQUIRED_PARTS)} frozen and unchanged; {len(REQUIRED_FILES)} required files present")

In [ ]:
!pip install -q -e "{REPO}[bench,merge,vlm]" 2>&1 | tail -2
!pip install -q -U colpali-engine pypdfium2 2>&1 | tail -1
for package in EXTRA_PIP:
    !pip install -q "{package}" 2>&1 | tail -1
# Kaggle preinstalls torchao 0.10; peft refuses to work while an incompatible torchao is importable
!pip uninstall -y -q torchao
os.makedirs(RESULTS, exist_ok=True)
json.dump({"batch": NAME, "jobs": JOBS}, open(f"{WORK}/jobs.json", "w"), indent=1)

In [ ]:
# Resume: a previous run of this notebook added as input (Add data -> Notebook output) gives back its
# finished shards; encode_unit.py skips every shard that is already there.
import glob, os, shutil, zipfile
_found = 0
for _z in sorted(glob.glob(f"/kaggle/input/**/results_{NAME}.zip", recursive=True)):
    with zipfile.ZipFile(_z) as _f:
        for _m in _f.infolist():
            _t = os.path.join(WORK, _m.filename)
            if _m.is_dir() or ".." in _m.filename or os.path.exists(_t) or _m.filename.endswith((".tmp", ".tmp.npz")):
                continue
            os.makedirs(os.path.dirname(_t), exist_ok=True)
            with _f.open(_m) as _src, open(_t, "wb") as _dst:
                shutil.copyfileobj(_src, _dst)
            _found += 1
for _d in sorted(glob.glob(f"/kaggle/input/**/results_{NAME}", recursive=True)):
    shutil.copytree(_d, RESULTS, dirs_exist_ok=True)
    _found += 1
print(f"resume: {_found} file(s) or folder(s) from earlier runs" if _found else "resume: starting fresh")

In [ ]:
cmd = [sys.executable, f"{REPO}/scripts/easy_validation/encode_unit.py", f"{WORK}/jobs.json", "--out", RESULTS,
       "--data", "/tmp/optdata", "--deadline-hours", str(DEADLINE_HOURS)]
subprocess.run(cmd, check=False)

In [ ]:
!pip freeze > {RESULTS}/pip_freeze.txt
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv > {RESULTS}/gpu.txt
!cd {REPO} && git rev-parse HEAD > {RESULTS}/commit.txt
!cat {RESULTS}/batch_summary.json
# store, do not deflate: float16 vectors barely compress; the folder goes once the archive tests sound
!cd {WORK} && rm -f results_{NAME}.zip && zip -q -r -0 results_{NAME}.zip results_{NAME} && zip -T -q results_{NAME}.zip && rm -rf results_{NAME} && ls -la results_{NAME}.zip